In [ ]:
import xml.etree.ElementTree as ET
import os
import pandas as pd
import ndjson
import subprocess

In [ ]:
subprocess.run(["git", "clone", "https://github.com/dracor-org/dutchdracor.git"], check=True)

In [2]:
list_of_titles = []
home_dir = 'dutchdracor/tei'

for title in os.listdir(home_dir):
    list_of_titles.append(title)
print(len(list_of_titles), "plays have been selected for analysis.")

98 plays have been selected for analysis.


In [4]:
def parse_play(tree):
    root_play = tree.getroot()
    play_title = root_play.find(".//{http://www.tei-c.org/ns/1.0}titleStmt/{http://www.tei-c.org/ns/1.0}title").text
    play_id = root_play.get("{http://www.w3.org/XML/1998/namespace}id")

# Create speakers dictionary
    speakers_dict = {}
    speaker_list = [persona for persona in root_play.findall(".//{http://www.tei-c.org/ns/1.0}personGrp") + root_play.findall(".//{http://www.tei-c.org/ns/1.0}person")]
    for speaker in speaker_list:
        speaker_id = speaker.get("{http://www.w3.org/XML/1998/namespace}id")
        if speaker.find(".//{http://www.tei-c.org/ns/1.0}name") != None:
          name = speaker.find(".//{http://www.tei-c.org/ns/1.0}name").text
        else:
          name = speaker.find(".//{http://www.tei-c.org/ns/1.0}persName").text
        gender = speaker.get("sex")
        speakers_dict[speaker_id] = {"name": name, "gender": gender}

    # Collect speeches
    speech_elements = root_play.findall(".//{http://www.tei-c.org/ns/1.0}sp")
    speeches = []
    for speech in speech_elements:
        speaker_id = speech.get("who").strip("#")
        speaker_info = speakers_dict.get(speaker_id, {"name": "Unknown Speaker", "gender": "Unknown"})
        speaker_name = speaker_info["name"]
        speaker_gender = speaker_info["gender"]
        lines = speech.findall(".//{http://www.tei-c.org/ns/1.0}lb") + speech.findall(".//{http://www.tei-c.org/ns/1.0}l") + speech.findall(".//{http://www.tei-c.org/ns/1.0}s")
        lines = [line.text for line in lines]
        speeches.append({'speaker': speaker_name, 'gender': speaker_gender, 'speech': lines, 'play': play_title, 'play_id': play_id})

    return speeches

acts = []
for title in list_of_titles:
    filename = title
    tree = ET.parse(home_dir + '/' + filename)
    play_speeches = parse_play(tree)
    acts.extend(play_speeches)

In [5]:
for act in acts:
    act['speech'] = [item for item in act['speech'] if item is not None]

In [ ]:
df_acts = pd.DataFrame(acts)
df_acts.head()

,speaker,gender,speech,play,play_id
0,AGAMÉMNON,MALE,"[Ia, Agamémnon is ’t, uw Vórst, die u komt wék...",Ifigenia,dut000147
1,ARKAS,MALE,"[Myn Heer, hoe! zyt gy ’t zelf? wat zaaken van...",Ifigenia,dut000147
2,AGAMÉMNON,MALE,"[Gelukkig is hy, die vernoeging weet te vinden...",Ifigenia,dut000147
3,ARKAS,MALE,[Sint wat tyd spreekt gy zó? hoe komt in uw’ g...,Ifigenia,dut000147
4,AGAMÉMNON,MALE,"[Gy zult niet stérven, neen; ik kan het niet g...",Ifigenia,dut000147


In [6]:
with open('speech_gender.ndjson', 'w') as fout:
	ndjson.dump(df_acts.to_dict('records'), fout)